In [1]:
import numpy as np
import tinker
from tinker import types

In [2]:
# Creating the training client and tokenizer

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(
    base_model="meta-llama/Llama-3.2-1B",
    rank=32,
)
tokenizer = training_client.get_tokenizer()

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Defining the training data

examples = [
    {"input": "banana split",      "output": "anana-bay plit-say"},
    {"input": "quantum physics",   "output": "uantum-qay ysics-phay"},
    {"input": "donut shop",        "output": "onut-day op-shay"},
    {"input": "pickle jar",        "output": "ickle-pay ar-jay"},
    {"input": "space exploration", "output": "ace-spay exploration-way"},
    {"input": "rubber duck",       "output": "ubber-ray uck-day"},
    {"input": "coding wizard",     "output": "oding-cay izard-way"},
]

In [4]:
# Tokenizing the training data: the model will learn to predict the output given the input (NOT THE INPUT)


def process_example(example, tokenizer):
    prompt = f"English: {example['input']}\nPig Latin:"

    # Tokenize prompt — the model sees this but is NOT trained on it
    # Assign zero weights to prompt tokens so they don't contribute to the loss
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    prompt_weights = [0] * len(prompt_tokens)

    # Tokenize completion — the model IS trained to produce this
    # Assign weights of 1 to completion tokens so they contribute to the loss
    completion_tokens = tokenizer.encode(
        f" {example['output']}\n\n", add_special_tokens=False
    )
    completion_weights = [1] * len(completion_tokens)

    # Concatenate and shift for next-token prediction
    tokens = prompt_tokens + completion_tokens
    weights = prompt_weights + completion_weights

    # Shift tokens and weights for next-token prediction
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    weights = weights[1:]

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs=dict(weights=weights, target_tokens=target_tokens),
    )


processed = [process_example(ex, tokenizer) for ex in examples]

In [5]:
# Training the model

for step in range(6):
    # Forward + backward: compute gradients on Tinker's GPUs
    fwdbwd_future = training_client.forward_backward(
        processed, "cross_entropy"
    )

    # Optimizer step: update the LoRA adapter weights
    optim_future = training_client.optim_step(
        types.AdamParams(learning_rate=1e-4)
    )

    # Wait for results and compute loss
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    logprobs = np.concatenate(
        [out['logprobs'].tolist()
         for out in fwdbwd_result.loss_fn_outputs]
    )
    weights = np.concatenate(
        [ex.loss_fn_inputs['weights'].tolist()
         for ex in processed]
    )
    loss = -np.dot(logprobs, weights) / weights.sum()
    print(f"Step {step}: loss = {loss:.4f}")


Step 0: loss = 6.4281
Step 1: loss = 5.7421
Step 2: loss = 4.6488
Step 3: loss = 3.4991
Step 4: loss = 2.5394
Step 5: loss = 1.9957


In [6]:
# Sample from the Fine-Tuned Model

sampler = training_client.save_weights_and_get_sampling_client(
    name="pig-latin-model"
)

prompt = types.ModelInput.from_ints(
    tokenizer.encode("English: coffee break\nPig Latin:")
)
params = types.SamplingParams(
    max_tokens=20, temperature=0.0, stop=["\n"]
)
result = sampler.sample(
    prompt=prompt, sampling_params=params, num_samples=8
).result()

print("Responses:")
for i, seq in enumerate(result.sequences):
    print(f"  {i}: {tokenizer.decode(seq.tokens)}")

Responses:
  0:  oof-bey  breakay


  1:  oof-bey  breakay


  2:  oof-bey  breakay


  3:  oof-bey  breakay


  4:  oof-bey  breakay


  5:  oof-bey  breakay


  6:  oof-bey  breakay


  7:  oof-bey  breakay


